# FT scripts: the Gate G dry run (Design 7.3, decisions D170 to D178)

Run by a person who is **not** in the HT-1 pair: this notebook writes model output (D150).

Order: setup, Cell 0, self checks, data, train `ft-dry`, reload, Gate G report.
The data step needs no GPU (CPU runtime is enough; set `NEED_GPU = False` in Cell 0).
Everything from the training step on needs an **L4**. Run the setup cell and Cell 0 again on every new runtime.

In [ ]:
# Setup: Drive, repo, secrets, installs. Run on every new runtime.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
REPO = '/content/banglaindoorvqa-main'
ROOT = '/content/drive/MyDrive/BanglaIndoorVQA'
token = userdata.get('GITHUB_TOKEN')

def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit('git failed: ' + (r.stderr or r.stdout).replace(token, '***')[-400:])
    return r.stdout

if not os.path.isdir(REPO):
    git('clone', f'https://{token}@github.com/blkeyd/banglaindoorvqa-main.git', REPO)
git('-C', REPO, 'pull')
print('repo at commit', git('-C', REPO, 'rev-parse', '--short', 'HEAD').strip())
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.chdir(REPO)
# D152, D153, D171: the same three pins in every session that loads an open model
!pip install -q flash-linear-attention transformers==5.18.0 peft==0.21.0

In [ ]:
# Cell 0: preflight. One PASS or FAIL line per input. Fix every FAIL before the next cell.
NEED_GPU = True          # False while you only run the data step on a CPU runtime
import csv, hashlib, importlib.util, json, os, shutil, subprocess, sys
from importlib import metadata
sys.path.insert(0, REPO + '/src')
fails = []

def line(ok, what, hint=''):
    print(('PASS  ' if ok else 'FAIL  ') + what + ('' if ok or not hint else '   -> ' + hint))
    if not ok:
        fails.append(what)

def note(text):
    print('NOTE  ' + text)

def attempt(what, fn, hint=''):
    """fn returns (ok, detail). An error inside it is a FAIL, never a crash."""
    try:
        ok, detail = fn()
    except Exception as exc:
        ok, detail = False, f'{type(exc).__name__}: {str(exc)[:150]}'
    line(ok, what + (f' ({detail})' if detail else ''), hint)

def sha(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def ver(pkg):
    try:
        return metadata.version(pkg)
    except Exception:
        return 'not installed'

print('--- A. Repo ---')
delivered = {'src/finetune.py': '8c4719aeae57', 'scripts/ft_finetune.py': 'c2c5f9d84e04', 'src/harness.py': '0a11abd8a4ec',
             'configs/models.yaml': 'bb1de9d2566b', 'configs/finetune.yaml': '3a5368f0bb4a'}
for rel, want in delivered.items():
    attempt(f'repo {rel} is the delivered file', lambda rel=rel, want=want: (sha(f'{REPO}/{rel}')[:12] == want, f'sha256 {sha(REPO + "/" + rel)[:12]}, expected {want}'),
            'upload the file from the FT chat again (a later fix from the chat changes the expected value)')
for rel in ('src/prompts.py', 'src/images.py', 'src/normalise.py', 'src/cost_ledger.py', 'scripts/asset8_harness.py',
            'configs/conditions.yaml', 'configs/prompts.yaml'):
    line(os.path.exists(f'{REPO}/{rel}'), f'repo {rel}')
head = subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', '-C', REPO, 'status', '--porcelain'], capture_output=True, text=True).stdout.strip()
note(f'repo at commit {head or "unknown"}' + ('; no uncommitted change' if not dirty else
     '; NOT committed yet: ' + ', '.join(ln[3:] for ln in dirty.splitlines()[:6]) + ' (commit before a seed is trained)'))
def _harness():
    import harness as H
    models = H.load_models(REPO + '/configs/models.yaml')
    h = H.canonical_sha(H.hf_settings(models['qwen']))
    ok = H.HARNESS_VERSION == '2026-10-07a' and all(k in models for k in ('qwen_ft_dry', 'qwen_ft_s42', 'qwen_ft_s43', 'qwen_ft_s44')) and h.startswith('6ca7d8af597c')
    return ok, f'version {H.HARNESS_VERSION}, base settings hash {h[:12]}'
attempt('harness 2026-10-07a, four adapter entries, base settings hash 6ca7d8af597c', _harness)
def _ft():
    import finetune as F
    ft = F.load_ft(REPO + '/configs/finetune.yaml')
    return True, f'FT scripts {F.FT_VERSION}, base model {ft["base_model"]}, peft_version {ft["peft_version"]}'
attempt('configs/finetune.yaml holds the Design 7.3 values', _ft)

print('--- B. Drive ---')
def _dataset():
    data = json.load(open(f'{ROOT}/data/banglaindoorvqa.json', encoding='utf-8'))
    houses = {r['house_id'] for r in data}
    n_qa = sum(len(r['qa_pairs']) for r in data)
    return (len(data), len(houses), n_qa) == (4406, 68, 18276), f'{len(data)} images, {len(houses)} houses, {n_qa} QA pairs'
attempt('data/banglaindoorvqa.json is the FULL merged dataset: 4406 images, 68 houses, 18276 QA pairs', _dataset,
        'this must be the full file with all three splits, not a train or test part')
def _splits():
    rows = list(csv.DictReader(open(f'{ROOT}/data/splits.csv', encoding='utf-8-sig')))
    c = {s: sum(1 for r in rows if r['split'] == s) for s in ('train', 'val', 'test')}
    return c == {'train': 43, 'val': 10, 'test': 15}, f'{c}'
attempt('data/splits.csv: 43 train, 10 val and 15 test houses', _splits)
def _rtest():
    n = len(json.load(open(f'{ROOT}/probes/R-TEST.json', encoding='utf-8'))['items'])
    return n == 1449, f'{n} items'
attempt('probes/R-TEST.json: 1449 items', _rtest)
line(os.path.exists(f'{ROOT}/probes/P-CORE.json'), 'probes/P-CORE.json (for the comparison with the earlier smoke run)')
def _images():
    folders = [d for d in os.listdir(f'{ROOT}/data/images') if os.path.isdir(f'{ROOT}/data/images/{d}')]
    return len(folders) == 68, f'{len(folders)} house folders'
attempt('data/images/: one folder per house, 68 folders', _images)
def _manifest():
    rows = list(csv.DictReader(open(f'{ROOT}/data/images_1024/images_manifest.csv', encoding='utf-8-sig')))
    n = len({r['image_file'] for r in rows})
    return n >= 1588, f'{n} prepared images listed; 4406 after the data step'
attempt('data/images_1024/images_manifest.csv: at least the 1588 images prepared so far', _manifest)
def _smoke():
    d = f'{ROOT}/outputs/smoke/t1/qwen'
    n = sum(1 for ln in open(f'{d}/cache.jsonl', encoding='utf-8') if ln.strip())
    stored = json.load(open(f'{d}/stage_state.json')).get('settings_sha256', '')
    return n >= 40 and stored.startswith('6ca7d8af597c'), f'{n} answers, settings hash {stored[:12]}'
attempt('outputs/smoke/t1/qwen: the earlier Qwen smoke run, 40 answers, settings hash 6ca7d8af597c', _smoke,
        'if the Qwen smoke run has another folder name, tell the FT chat (reload then needs --base-old)')
def _write():
    os.makedirs(f'{ROOT}/models', exist_ok=True)
    p = f'{ROOT}/models/_preflight.tmp'
    open(p, 'w').write('x')
    os.remove(p)
    return True, ''
attempt('Drive is writable (models/)', _write)
for rel in ('models/ft-dry', 'outputs/smoke/ft-dry', 'outputs/smoke/ft-base'):
    if os.path.exists(f'{ROOT}/{rel}'):
        note(f'Drive {rel} already exists: an earlier dry run. The commands continue it; train --restart starts again.')

print('--- C. Session ---')
line(ver('transformers') == '5.18.0', f"transformers is 5.18.0 (found {ver('transformers')})", 'run the setup cell (D153)')
line(ver('peft') == '0.21.0', f"peft is 0.21.0 (found {ver('peft')})", 'run the setup cell (D171)')
line(importlib.util.find_spec('fla') is not None, 'flash-linear-attention is installed (D152)', 'run the setup cell')
line(bool(os.environ.get('HF_TOKEN')), 'HF_TOKEN is set from Colab Secrets')
free = shutil.disk_usage('/content' if os.path.isdir('/content') else '/').free / 2 ** 30
line(free >= 25, f'local disk: {free:.0f} GB free (25 needed for the model and the image copies)')
if NEED_GPU:
    def _hub():
        from huggingface_hub import HfApi
        commit = HfApi().model_info('Qwen/Qwen3.5-4B', token=os.environ.get('HF_TOKEN')).sha
        if not commit.startswith('851bf6e806ef'):
            note(f'Qwen/Qwen3.5-4B has a newer commit ({commit[:12]}) than the smoke run of 5 October (851bf6e806ef): '
                 'check H2 of the Gate G report may then differ. Tell the FT chat.')
        return True, f'commit {commit[:12]}'
    attempt('the token can read Qwen/Qwen3.5-4B on Hugging Face', _hub)
    def _gpu():
        import torch
        name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU'
        if torch.__version__ != '2.11.0+cu130':
            note(f'torch is {torch.__version__}; the smoke run of 5 October used 2.11.0+cu130. Check H2 may then differ.')
        return 'L4' in name, name
    attempt('the GPU is an L4', _gpu, 'Runtime, Change runtime type, L4 GPU')
else:
    note('no GPU and no Hugging Face check: NEED_GPU is False (enough for the data step)')

print()
print('ALL PASS: go on to the next cell.' if not fails else f'{len(fails)} FAIL: fix them before the next cell.')

In [ ]:
# 1. Self checks. Expected last lines: 14/14, 13/13, 32/32, then 44/44 (every line OK).
!python src/prompts.py | tail -1
!python src/images.py | tail -1
!python scripts/asset8_harness.py selftest | tail -1
!python scripts/ft_finetune.py selftest

In [ ]:
# 2. Data (no GPU needed): splits, checks against Design 3.1 and R-TEST, the validation list,
#    and the prepared copy of every image (D175). About 2,800 images are new; allow 30 to 45 minutes.
#    Safe to run again: it only prepares what is missing.
!python scripts/ft_finetune.py data --prepare-images all

After the data step: commit `probes/FT-VAL400.ids.txt` (qa_ids only) the way the six probe id lists were committed.
A copy is in Drive `data/ft/`. If the runtime is changed before the commit, run the data step again; it rewrites the file.

In [ ]:
# 3. The Gate G dry run (L4): 100 training examples, 1 epoch, then the answers from memory
#    and the overfit probe. If the session ends, run the same cell again. To repeat it: add --restart
!python scripts/ft_finetune.py train --run ft-dry

In [ ]:
# 4. The harness part of Gate G (L4): the saved adapter and the base model on the same 20 R-TEST
#    items, then the base model on the items of the earlier smoke run. Three model loads.
!python scripts/ft_finetune.py reload --run ft-dry

In [ ]:
# 5. The Gate G report. Copy the whole output into the FT chat.
!python scripts/ft_finetune.py gate-g --run ft-dry

In [ ]:
# Optional, only for a person outside the HT-1 pair: 10 of the 20 reloaded answers (model output).
# !python scripts/asset8_harness.py peek --stage smoke --run-name ft-dry --model qwen_ft_dry --n 10 --i-am-not-in-the-ht1-pair